# Realized-volatility benchmark — HAR-RV · N-HAR · ModernTCN · FiLM-TCN

Runs **four models × five FX pairs × two horizons (h = 1, h = 5)** and collects
one comparison table.

| Model | What it is | Driver |
|---|---|---|
| **HAR-RV** | Corsi (2009) heterogeneous autoregression on ln RV, OLS | `HAR_RV_run.py` |
| **N-HAR** | HAR + weekday×RV_d controls + LASSO-selected macro-release dummies over the forecast window (Plihal) | `HAR_X_run.py` |
| **ModernTCN** | ModernTCN backbone, price history only — no event information | `run.py` |
| **FiLM-TCN** | same backbone + news conditioning: past releases flow through the backbone as an extra channel, and the *known* future release schedule FiLM-conditions the head | `run.py --use_events` |

**Pairs** `AUDUSD, EURUSD, GBPUSD, USDCHF, USDJPY` — each read from
`data/<PAIR>_lnRV.csv` and paired by name with its own calendar
`data/<PAIR>_EVENTS.csv`.

**Target** for horizon *h* is the log of the realized variance averaged over the
forecast window, $Y_t^{(h)} = \ln\!\big(\frac1h\sum_{k=1}^{h} RV_{t+k}\big)$ — the
deep models get it through `--aggregate_horizon --pred_len h` (a log-sum-exp over
the $\ln RV$ window, less $\ln h$), so all four rows predict the same quantity.
This is *not* the forward mean of logs $\frac1h\sum_k \ln RV_{t+k}$ used
previously: averaging the variance and then logging weights a turbulent day
inside the window far more heavily than averaging logs does. At $h=1$ the two
coincide, so only the $h=5$ rows move.

**Split** — deep models: train ≤ 2021, validate 2022–23, test ≥ 2024.
HAR-RV / N-HAR fold the validation window into training (OLS/LASSO tune nothing
on it) and score the identical 2024–25 test window.

**Metrics** — MSE and MAE on the ln RV scale; QLIKE (Patton, 2011) on the
variance scale after exponentiation. All three scripts use the same
definitions (`utils/metrics.py`), so the rows are directly comparable.

**Cost** — the two linear models take seconds. The two deep models train
`ITR` seeds each (default 5, seeds 2021…2025) for 2 models × 5 pairs × 2
horizons = 20 configurations ≈ 100 trainings. On a Colab T4 that is roughly
1–2 hours; set `QUICK_TEST = True` in the config cell for a ~3-minute
end-to-end rehearsal first.

> **Set the runtime to GPU**: *Runtime → Change runtime type → T4 GPU*.
> Everything also runs on CPU, just slower.

Every configuration caches its result to `benchmark_h1_h5/runs/*.json`, so if
the Colab session drops you can re-run the cells and only the missing
configurations are trained again.

## 1 · Repository

In [ ]:
# Clone (or refresh) the repo on Colab; use the current checkout anywhere else.
import os, sys, subprocess, pathlib

REPO_URL  = "https://github.com/Mr0022/javad.git"
BRANCH    = "correct_log"
CLONE_DIR = "/content/javad"          # Colab only

IN_COLAB = "google.colab" in sys.modules


def sh(*cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(list(cmd), check=check)


if IN_COLAB:
    if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
        # already cloned in this session -> pull the latest branch tip
        sh("git", "-C", CLONE_DIR, "fetch", "--depth", "1", "origin", BRANCH)
        sh("git", "-C", CLONE_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD")
    else:
        sh("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR)
    REPO = CLONE_DIR
else:
    # running from a local checkout: walk up until run.py is found
    REPO = os.path.abspath(os.getcwd())
    while REPO != "/" and not os.path.isfile(os.path.join(REPO, "run.py")):
        REPO = os.path.dirname(REPO)

os.chdir(REPO)
assert os.path.isfile("run.py"), f"run.py not found from {os.getcwd()}"


def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else "(no git metadata)"


print("\nrepo   :", REPO)
print("branch :", git("rev-parse", "--abbrev-ref", "HEAD"))
print("commit :", git("log", "-1", "--oneline"))
print("series :", sorted(f for f in os.listdir("data") if f.endswith("_lnRV.csv")))
print("events :", sorted(f for f in os.listdir("data") if f.endswith("_EVENTS.csv")))

## 2 · Packages and device

In [ ]:
import importlib.util, subprocess, sys

PYPI = {"sklearn": "scikit-learn"}
missing = [m for m in ("torch", "numpy", "pandas", "scipy",
                       "statsmodels", "sklearn", "matplotlib")
           if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[PYPI.get(m, m) for m in missing]], check=True)
else:
    print("all packages already present")

import torch
if torch.cuda.is_available():
    print(f"\ntorch {torch.__version__}  |  GPU: {torch.cuda.get_device_name(0)}")
else:
    print(f"\ntorch {torch.__version__}  |  CPU only — the deep runs will be slow.")
    print("Colab: Runtime -> Change runtime type -> T4 GPU")

## 3 · Experiment configuration

`QUICK_TEST = True` shrinks the deep runs to 1 seed × 2 epochs so the whole
notebook finishes in a few minutes — use it to check the pipeline end to end,
then set it back to `False` for the real numbers.

In [ ]:
import pathlib

PAIRS    = ["AUDUSD", "EURUSD", "GBPUSD", "USDCHF", "USDJPY"]
HORIZONS = [1, 5]

# --- deep models (ModernTCN, FiLM-TCN) --------------------------------------
ITR          = 5      # seeds 2021 .. 2021+ITR-1, as in scripts/moderntcn.sh
TRAIN_EPOCHS = 40
PATIENCE     = 8
NUM_WORKERS  = 0

# --- N-HAR -------------------------------------------------------------------
# 'min' = the CV minimum (the faithful reading of Plihal, and HAR_X_run.py's
# default); '1se' = the one-standard-error rule, a more conservative robustness
# check that shrinks harder.
LAMBDA_RULE = "min"

QUICK_TEST = False          # <- True for a ~3 minute rehearsal of every cell
if QUICK_TEST:
    ITR, TRAIN_EPOCHS, PATIENCE = 1, 2, 2

FORCE_RERUN = False         # True = ignore cached results and re-run everything
# NOTE: cached runs are keyed by (model, pair, horizon) only, not by the target
# definition. Any benchmark_h1_h5/runs/*.json written before the target became
# ln((1/h) * sum RV) holds numbers from the old mean-of-logs target and would be
# silently mixed into the table -- delete them, or set FORCE_RERUN = True once.

OUT  = pathlib.Path("benchmark_h1_h5")
RUNS = OUT / "runs"         # one json per (model, pair, horizon)
LOGS = OUT / "logs"         # full stdout of every subprocess
for d in (OUT, RUNS, LOGS):
    d.mkdir(parents=True, exist_ok=True)

print(f"{len(PAIRS)} pairs x {len(HORIZONS)} horizons x 4 models = "
      f"{len(PAIRS) * len(HORIZONS) * 4} table cells")
print(f"deep training runs: {len(PAIRS) * len(HORIZONS) * 2} configs x {ITR} "
      f"seeds = {len(PAIRS) * len(HORIZONS) * 2 * ITR} trainings "
      f"({TRAIN_EPOCHS} epochs max, patience {PATIENCE})")
print("output ->", OUT.resolve())

## 4 · Hyper-parameters

**One fixed configuration for every deep run.** All four deep cells
(ModernTCN and FiLM-TCN, h = 1 and h = 5) share the block below; the only
things that differ are `--pred_len` and the event flags. That makes
ModernTCN vs FiLM-TCN a **controlled ablation of the event path** — identical
backbone, identical optimiser, identical look-back — rather than a comparison
of two separately tuned models.

The block is the `ModernTCN1` Optuna optimum with `dropout` raised to 0.5,
written out literally so it can be edited in one place. Set
`HPARAM_SOURCE = "tuned"` to switch back to reading
`tuningresults/<study>/best_params.json` per model and horizon.

Scalars expand into 4-stage lists the way `tune.py` does it
(`dims = dw_dims = [dim]*4`, `num_blocks = [nb]*4`, and
`patch_stride = min(patch_stride, patch_size)` — `tune.py:150`).

⚠️ This configuration was tuned for EUR/USD at h = 1. Holding it fixed is what
buys the controlled comparison, but no row is then a per-pair or per-horizon
optimum — h = 5 in particular runs at an h = 1 setting. Re-tune with `tune.py`
before reading any single row as final.

In [ ]:
import json

# ---- the configuration every deep run uses --------------------------------
HPARAMS = dict(
    seq_len       = 70,
    patch_size    = 16,
    patch_stride  = 8,
    ffn_ratio     = 2,
    num_blocks    = 2,
    large_size    = 27,
    small_size    = 5,
    dim           = 32,
    dropout       = 0.5,
    head_dropout  = 0.13413677333143775,
    revin         = 1,
    learning_rate = 0.0063484758647924695,
    batch_size    = 256,
    event_dim     = 16,          # FiLM-TCN only
)
EVENT_FUSION = "channel"

# "fixed" -> HPARAMS above for every (model, horizon)
# "tuned" -> the per-study Optuna optimum in tuningresults/ instead
HPARAM_SOURCE = "fixed"

STUDY = {("ModernTCN", 1): "ModernTCN1", ("ModernTCN", 5): "ModernTCN5",
         ("FiLM-TCN",  1): "EVENTTCN1",  ("FiLM-TCN",  5): "EVENTTCN5"}


def expand(p):
    # params dict -> CLI flags, expanded the way tune.py expands a trial
    stride = min(p["patch_stride"], p["patch_size"])    # tune.py:150 clamps this
    d, ls, ss, nb = p["dim"], p["large_size"], p["small_size"], p["num_blocks"]
    flags = ["--seq_len", p["seq_len"],
             "--patch_size", p["patch_size"], "--patch_stride", stride,
             "--ffn_ratio", p["ffn_ratio"],
             "--num_blocks", nb, nb, nb, nb,
             "--large_size", ls, ls, ls, ls,
             "--small_size", ss, ss, ss, ss,
             "--dims", d, d, d, d,
             "--dw_dims", d, d, d, d,
             "--dropout", p["dropout"], "--head_dropout", p["head_dropout"],
             "--revin", p["revin"],
             "--learning_rate", p["learning_rate"],
             "--batch_size", p["batch_size"]]
    return [str(x) for x in flags]


def hparams_for(model, h):
    # (CLI flags, params) for one deep configuration
    if HPARAM_SOURCE == "tuned":
        study = STUDY[(model, h)]
        p = json.load(open(f"tuningresults/{study}/best_params.json"))["params"]
    else:
        p = dict(HPARAMS)
    return expand(p), p


keys = ["seq_len", "patch_size", "patch_stride", "dim", "num_blocks",
        "large_size", "small_size", "ffn_ratio", "dropout", "head_dropout",
        "learning_rate", "batch_size", "revin", "event_dim"]
print(f"hyper-parameters: {HPARAM_SOURCE}\n")
print(f"{'model':<12}{'h':>3}  " + "".join(f"{k:>14}" for k in keys))
print("-" * (15 + 14 * len(keys)))
for model in ("ModernTCN", "FiLM-TCN"):
    for h in (1, 5):
        _, p = hparams_for(model, h)
        cells = []
        for k in keys:
            v = "-" if (k == "event_dim" and model == "ModernTCN") else p.get(k, "-")
            cells.append(f"{v:>14.6g}" if isinstance(v, float) else f"{str(v):>14}")
        print(f"{model:<12}{h:>3}  " + "".join(cells))

## 5 · Subprocess runner

Every model is driven through its own script in a fresh process — the same
commands `scripts/moderntcn.sh` and `scripts/eventtcn.sh` use, so the notebook
cannot silently diverge from the repo. Full stdout goes to
`benchmark_h1_h5/logs/`; the notebook only echoes progress. Each seed's
`mse: … mae: … rse: … qlike: …` line is parsed and averaged across seeds.

In [ ]:
import re, time, subprocess, json, numpy as np

METRIC_RE = re.compile(r"mse:\s*([-\d.eE+]+),\s*mae:\s*([-\d.eE+]+),"
                       r"\s*rse:\s*[-\d.eE+]+,\s*qlike:\s*([-\d.eE+]+)")
NTEST_RE  = re.compile(r"^test (\d+)$", re.M)


def run_cmd(cmd, log_path, cwd=None):
    """Run a command, tee stdout to log_path, echo a compact progress trace."""
    t0, lines = time.time(), []
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            lines.append(line)
            s = line.rstrip()
            if s.startswith("Epoch:") and "Steps:" in s:
                print(".", end="", flush=True)          # heartbeat, one per epoch
            elif s.startswith("mse:"):
                print(" " + s, flush=True)
            elif (s.startswith(">>>>>>> run") or "Early stopping" in s
                  or s.startswith("Use GPU") or s.startswith("Use CPU")):
                print("  " + s.strip(">< "), end=" ", flush=True)
        proc.wait()
    out = "".join(lines)
    if proc.returncode != 0:
        print("".join(lines[-25:]))
        raise RuntimeError(f"{cmd[1]} exited {proc.returncode}; see {log_path}")
    return out, time.time() - t0


def cache_path(model, pair, h):
    return RUNS / f"{model.replace('/', '-')}_{pair}_h{h}.json"


def load(model, pair, h):
    """The stored result for a cell, or None if it has never been computed."""
    p = cache_path(model, pair, h)
    return json.loads(p.read_text()) if p.exists() else None


def cached(model, pair, h):
    """load(), but None while FORCE_RERUN asks for everything to be redone."""
    return None if FORCE_RERUN else load(model, pair, h)


def store(rec):
    cache_path(rec["model"], rec["pair"], rec["horizon"]).write_text(
        json.dumps(rec, indent=2))
    return rec


def summarise(rec):
    sd = f" +/- {rec['MSE_std']:.4f}" if rec.get("MSE_std") is not None else ""
    print(f"  => {rec['model']:<10} {rec['pair']} h={rec['horizon']}:  "
          f"MSE {rec['MSE']:.4f}{sd}   MAE {rec['MAE']:.4f}   "
          f"QLIKE {rec['QLIKE']:.4f}   (n_test={rec['n_test']}, "
          f"{rec['n_seeds']} seed(s), {rec['seconds']:.0f}s)")

## 6 · HAR-RV  (Corsi 2009 — OLS)

`HAR_RV_run.py` always estimates h = 1, 5 and 22; we keep h = 1 and 5. Each
pair runs in its own working directory so the script's figures and parameter
tables (`HAR-RV results/`) are kept per pair instead of overwriting each other.

In [ ]:
import pandas as pd

HAR_RV_DIR = OUT / "har_rv"


def har_rv_ntest(out):
    """Test-window row count per horizon, from the script's split table."""
    n = {}
    for block in re.split(r"HORIZON\s+h\s*=\s*", out)[1:]:
        m = re.search(r"^\s+Test\s+(\d+)\s", block, re.M)
        if m:
            n[int(block.split()[0])] = int(m.group(1))
    return n


todo = [p for p in PAIRS
        if any(cached("HAR-RV", p, h) is None for h in HORIZONS)]
print(f"HAR-RV: {len(todo)} of {len(PAIRS)} pairs to fit "
      f"({len(PAIRS) - len(todo)} cached)\n")

for pair in todo:
    wd = HAR_RV_DIR / pair
    wd.mkdir(parents=True, exist_ok=True)
    print(f"[run] HAR-RV {pair}", end=" ", flush=True)
    out, secs = run_cmd([sys.executable, str(pathlib.Path(REPO) / "HAR_RV_run.py"),
                         "--data_path",
                         str(pathlib.Path(REPO) / "data" / f"{pair}_lnRV.csv")],
                        LOGS / f"HAR-RV_{pair}.log", cwd=wd)
    print(f"({secs:.0f}s)")

    n_test = har_rv_ntest(out)
    m = pd.read_csv(wd / "HAR-RV results" / "har_rv_all_metrics.csv")
    m = m[m["split"] == "test"]
    for h in HORIZONS:
        r = m[m["horizon"] == h].iloc[0]
        store({"model": "HAR-RV", "pair": pair, "horizon": h,
               "MSE": float(r.MSE), "MAE": float(r.MAE), "QLIKE": float(r.QLIKE),
               "MSE_std": None, "MAE_std": None, "QLIKE_std": None,
               "n_seeds": 1, "n_test": n_test.get(h),
               "seconds": secs / len(HORIZONS)})

print()
for pair in PAIRS:
    for h in HORIZONS:
        summarise(load("HAR-RV", pair, h))

## 7 · N-HAR  (HAR + weekday controls + LASSO-selected release dummies)

`HAR_X_run.py` estimates three nested models on an identical row sample:
**HAR** (the reference row, numerically the same fit as HAR-RV above),
**HAR+DOW** (weekday×RV_d — the control that stops "macro news helps" from
being a restatement of "Friday is busy") and **N-HAR**. N-HAR's headline gain
is defined *against HAR+DOW*, so HAR+DOW is kept in the saved results as a
control row even though it is not one of the four models in the table.

It also writes `nhar_selected.csv` (which release dummies LASSO kept, with
fold stability) and `har_x_losses.csv` (per-observation losses, ready for an
MCS / Diebold-Mariano test).

In [ ]:
HAR_X_DIR = OUT / "har_x"
need = any(cached(m, p, h) is None
           for m in ("N-HAR", "HAR+DOW") for p in PAIRS for h in HORIZONS)

if need:
    HAR_X_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[run] N-HAR: {len(PAIRS)} pairs x {len(HORIZONS)} horizons "
          f"(lambda rule '{LAMBDA_RULE}')", flush=True)
    _, secs = run_cmd([sys.executable, "HAR_X_run.py",
                       "--root_path", "./data",
                       "--pairs", *PAIRS,
                       "--horizons", *map(str, HORIZONS),
                       "--target", "ln_RV",
                       "--lambda_rule", LAMBDA_RULE,
                       "--output_dir", str(HAR_X_DIR)],
                      LOGS / "N-HAR_all_pairs.log")
    print(f"({secs:.0f}s)\n")

    mx = pd.read_csv(HAR_X_DIR / "har_x_metrics.csv")
    per_cell = secs / max(len(mx), 1)
    for _, r in mx.iterrows():
        if r["model"] == "HAR":            # identical to HAR-RV; used as a check
            continue
        store({"model": r["model"], "pair": r["pair"], "horizon": int(r["horizon"]),
               "MSE": float(r.MSE), "MAE": float(r.MAE), "QLIKE": float(r.QLIKE),
               "MSE_std": None, "MAE_std": None, "QLIKE_std": None,
               "n_seeds": 1, "n_test": int(r.n_test), "seconds": per_cell,
               "n_selected": (None if pd.isna(r.get("n_selected"))
                              else int(r["n_selected"]))})

    # cross-check: HAR_X_run.py's HAR row must reproduce HAR_RV_run.py
    print("consistency check — HAR (HAR_X_run.py) vs HAR-RV (HAR_RV_run.py):")
    for _, r in mx[mx["model"] == "HAR"].iterrows():
        ref = load("HAR-RV", r["pair"], int(r["horizon"]))
        if ref is None:
            print(f"  {r['pair']} h={int(r['horizon'])}: HAR-RV not computed "
                  f"yet — run the cell above first")
            continue
        d = abs(float(r.MSE) - ref["MSE"])
        print(f"  {r['pair']} h={int(r['horizon'])}: "
              f"{float(r.MSE):.6f} vs {ref['MSE']:.6f}  |diff| = {d:.2e}"
              f"  {'OK' if d < 1e-6 else '<-- differs'}")
else:
    print("[cached] N-HAR")

print()
for pair in PAIRS:
    for h in HORIZONS:
        for m in ("HAR+DOW", "N-HAR"):
            rec = load(m, pair, h)
            if rec:
                summarise(rec)

## 8 · Deep models — the runner

One `run.py` process per (model, pair, horizon); `--itr ITR` sweeps seeds
2021…2021+ITR-1 inside it and the reported number is the mean over seeds
(± sample std). `--model_id` carries the pair and horizon so each
configuration gets its own `checkpoints/` and `results/` directory instead of
overwriting the previous pair's.

Both models are called with the identical flag list from §4; FiLM-TCN only
appends `--use_events --event_dim 16 --event_fusion channel`. `run.py` then
switches the dataset to `custom_events` and picks the calendar paired with
the series by name (`AUDUSD_lnRV.csv → AUDUSD_EVENTS.csv`).

In [ ]:
def run_deep(model, pair, h):
    rec = cached(model, pair, h)
    if rec is not None:
        print(f"[cached] {model} {pair} h={h}")
        summarise(rec)
        return rec

    flags, p = hparams_for(model, h)
    tag = model.replace("-", "")
    cmd = [sys.executable, "run.py",
           "--is_training", "1",
           "--model_id", f"{tag}_{pair}_h{h}",
           "--model", "ModernTCN",
           "--data", "custom",
           "--root_path", "./data/", "--data_path", f"{pair}_lnRV.csv",
           "--features", "S", "--target", "ln_RV",
           "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
           "--aggregate_horizon", "--pred_len", str(h),
           *flags,
           "--use_multi_scale", "False", "--lradj", "TST", "--pct_start", "0.3",
           "--train_epochs", str(TRAIN_EPOCHS), "--patience", str(PATIENCE),
           "--num_workers", str(NUM_WORKERS), "--itr", str(ITR)]
    if model == "FiLM-TCN":
        cmd += ["--use_events", "--event_dim", str(p["event_dim"]),
                "--event_fusion", EVENT_FUSION]

    print(f"[run] {model} {pair} h={h}", flush=True)
    out, secs = run_cmd(cmd, LOGS / f"{model}_{pair}_h{h}.log")

    per_seed = np.array([[float(x) for x in m.groups()]
                         for m in METRIC_RE.finditer(out)])   # (seeds, 3)
    if len(per_seed) != ITR:
        raise RuntimeError(f"expected {ITR} metric lines, parsed {len(per_seed)} "
                           f"— see {LOGS / f'{model}_{pair}_h{h}.log'}")
    mean = per_seed.mean(0)
    std = ([float(v) for v in per_seed.std(0, ddof=1)] if ITR > 1
           else [None] * 3)
    n_test = NTEST_RE.search(out)

    rec = {"model": model, "pair": pair, "horizon": h,
           "MSE": float(mean[0]), "MAE": float(mean[1]), "QLIKE": float(mean[2]),
           "MSE_std": std[0], "MAE_std": std[1], "QLIKE_std": std[2],
           "n_seeds": int(ITR), "n_test": int(n_test.group(1)) if n_test else None,
           "seconds": secs,
           "per_seed_mse": per_seed[:, 0].tolist()}
    store(rec)
    summarise(rec)
    return rec


def run_grid(model):
    t0 = time.time()
    grid = [(p, h) for h in HORIZONS for p in PAIRS]
    for i, (pair, h) in enumerate(grid, 1):
        run_deep(model, pair, h)
        done = time.time() - t0
        if i < len(grid):
            print(f"      [{i}/{len(grid)} done, {done / 60:.1f} min elapsed, "
                  f"~{done / i * (len(grid) - i) / 60:.1f} min left]\n")
    print(f"\n{model}: {len(grid)} configurations in {(time.time() - t0) / 60:.1f} min")

## 9 · ModernTCN  (no events)

In [ ]:
run_grid("ModernTCN")

## 10 · FiLM-TCN  (`--use_events`)

In [ ]:
run_grid("FiLM-TCN")

## 11 · Results table

`benchmark_h1_h5/results_long.csv` holds one row per (model, pair, horizon);
`results_table.md` holds the rendered tables below.

In [ ]:
import pandas as pd, numpy as np

MODELS  = ["HAR-RV", "N-HAR", "ModernTCN", "FiLM-TCN"]
CONTROL = ["HAR+DOW"]

res = pd.DataFrame([json.loads(f.read_text()) for f in sorted(RUNS.glob("*.json"))])
res["model"] = pd.Categorical(res["model"], MODELS + CONTROL, ordered=True)
res = (res[res["horizon"].isin(HORIZONS) & res["pair"].isin(PAIRS)]
       .sort_values(["horizon", "pair", "model"]).reset_index(drop=True))
res.to_csv(OUT / "results_long.csv", index=False)


def md_block(metric):
    """Markdown tables of `metric`: rows = pairs (+ mean), cols = the 4 models."""
    lines = []
    for h in HORIZONS:
        piv = (res[res["horizon"] == h]
               .pivot_table(index="pair", columns="model", values=metric,
                            observed=True)
               .reindex(index=PAIRS, columns=MODELS))
        lines += [f"\n**{metric} — h = {h}**  (lower is better)\n",
                  "| Pair | " + " | ".join(MODELS) + " |",
                  "|---|" + "---|" * len(MODELS)]
        for pair in PAIRS:
            row = piv.loc[pair]
            cells = [f"{row[m]:.4f}" if pd.notna(row[m]) else "–" for m in MODELS]
            if row.notna().any():
                best = MODELS.index(row.astype(float).idxmin())
                cells[best] = f"**{cells[best]}**"
            lines.append(f"| {pair} | " + " | ".join(cells) + " |")
        lines.append("| **Mean** | " + " | ".join(
            f"{piv[m].mean():.4f}" if piv[m].notna().any() else "–"
            for m in MODELS) + " |")
    return "\n".join(lines)


md_all = ["# Benchmark — h = 1 and h = 5, five FX pairs\n",
          f"Deep models: mean over {ITR} seed(s), {TRAIN_EPOCHS} epochs max, "
          f"patience {PATIENCE}. N-HAR lambda rule: '{LAMBDA_RULE}'.",
          "Bold = best model in the row."]
for metric in ("MSE", "MAE", "QLIKE"):
    md_all.append(md_block(metric))

# --- the two event-vs-no-event contrasts this benchmark exists to show -------
md_all.append("\n**Effect of conditioning on the release calendar** "
              "(% change in test MSE, negative = the event model wins)\n")
md_all.append("| Pair | h | N-HAR vs HAR-RV | FiLM-TCN vs ModernTCN |")
md_all.append("|---|---|---|---|")


def pct_change(g, a, b):
    if a in g.index and b in g.index and pd.notna(g[a]) and pd.notna(g[b]):
        return 100.0 * (g[a] - g[b]) / g[b]
    return np.nan


deltas = []
for h in HORIZONS:
    block = []
    for pair in PAIRS:
        g = res[(res.horizon == h) & (res.pair == pair)].set_index("model")["MSE"]
        d = {"pair": pair, "horizon": h,
             "N-HAR vs HAR-RV": pct_change(g, "N-HAR", "HAR-RV"),
             "FiLM-TCN vs ModernTCN": pct_change(g, "FiLM-TCN", "ModernTCN")}
        block.append(d)
        md_all.append(f"| {pair} | {h} | {d['N-HAR vs HAR-RV']:+.1f}% | "
                      f"{d['FiLM-TCN vs ModernTCN']:+.1f}% |")
    b = pd.DataFrame(block)
    md_all.append(f"| **Mean** | **{h}** | **{b['N-HAR vs HAR-RV'].mean():+.1f}%** | "
                  f"**{b['FiLM-TCN vs ModernTCN'].mean():+.1f}%** |")
    deltas += block
dlt = pd.DataFrame(deltas)
dlt.to_csv(OUT / "event_effect_pct_mse.csv", index=False)

text = "\n".join(md_all) + "\n"
(OUT / "results_table.md").write_text(text)

try:                                  # rendered tables in Colab/Jupyter
    from IPython.display import Markdown, display
    display(Markdown(text))
except Exception:
    print(text)

print("\nwrote:", OUT / "results_long.csv", "|", OUT / "results_table.md",
      "|", OUT / "event_effect_pct_mse.csv")

## 12 · Figure

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"HAR-RV": "#1a1a2e", "N-HAR": "#2166ac",
          "ModernTCN": "#fc8d59", "FiLM-TCN": "#d73027"}

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6.2 * len(HORIZONS), 4.2),
                         sharey=False)
axes = np.atleast_1d(axes)
x, w = np.arange(len(PAIRS)), 0.8 / len(MODELS)

for ax, h in zip(axes, HORIZONS):
    sub = res[res["horizon"] == h]
    piv = (sub.pivot_table(index="pair", columns="model", values="MSE",
                           observed=True)
           .reindex(index=PAIRS, columns=MODELS))
    # seed dispersion, where there is any (deep models with ITR > 1)
    err = (sub.pivot_table(index="pair", columns="model", values="MSE_std",
                           observed=True)
           .reindex(index=PAIRS, columns=MODELS)
           if "MSE_std" in sub and sub["MSE_std"].notna().any()
           else piv * np.nan)
    for i, m in enumerate(MODELS):
        e = err[m].to_numpy(dtype=float)
        e = None if np.isnan(e).all() else np.nan_to_num(e, nan=0.0)
        ax.bar(x + (i - (len(MODELS) - 1) / 2) * w, piv[m], w, yerr=e,
               error_kw={"elinewidth": 0.9, "capsize": 2, "ecolor": "#444"},
               label=m, color=COLORS[m], edgecolor="white", linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(PAIRS, rotation=20)
    ax.set_title(f"h = {h}"); ax.set_ylabel("test MSE  (ln RV)")
    ax.grid(axis="y", alpha=0.3, linewidth=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

axes[0].legend(frameon=False, ncol=2, fontsize=9)
fig.suptitle("Out-of-sample MSE (test 2024-2025) by pair and horizon"
             + (f"  —  deep models: mean +/- std over {ITR} seeds" if ITR > 1 else ""),
             y=1.02)
fig.tight_layout()
fig.savefig(OUT / "fig_mse_by_pair.png", dpi=200, bbox_inches="tight")
plt.show()
print("wrote:", OUT / "fig_mse_by_pair.png")

## 13 · Reading the numbers

* **Seeds.** Deep rows are the mean of `ITR` seeds; `results_long.csv` carries
  the per-seed MSEs and the sample std. A gap between ModernTCN and FiLM-TCN
  smaller than the seed std is noise, not an architecture effect.
* **Tuning.** Every deep run uses the single fixed block in §4, so
  FiLM-TCN differs from ModernTCN only by the event path — the MSE gap is
  attributable to the release calendar, not to a different backbone. The
  price is that no row is tuned for its own pair or horizon: the block is an
  EUR/USD h=1 optimum, so h=5 especially is running off-optimum and both deep
  rows may sit above what a per-cell `tune.py` sweep would reach. Set
  `HPARAM_SOURCE = "tuned"` in §4 for the per-study optima instead — faster
  models, but the ablation stops being controlled.
* **Test samples.** `n_test` is reported per row, and the two families differ
  by exactly one observation at every (pair, horizon). `Dataset_Custom`
  back-fills the deep models' test slice by `seq_len`
  (`data_loader.py:81`), so they lose nothing to the look-back and score
  every 2024–25 target. HAR-RV / N-HAR index rows by the *predictor* date and
  drop the trailing rows whose forward target is incomplete, which costs them
  the first 2024 target — its predictor row is the last 2023 trading day, so
  it falls on the training side of their split. Same window, one row of
  offset; it is a property of the repo's protocol, not of this notebook.
* **N-HAR selection.** `benchmark_h1_h5/har_x/nhar_selected.csv` lists the
  release dummies LASSO kept per pair and horizon, with fold stability —
  the direct read of *which* announcements carry the signal. Re-run §7 with
  `LAMBDA_RULE = "1se"` for the conservative check.
* **Significance.** `har_x/har_x_losses.csv` holds per-observation squared,
  absolute and QLIKE losses for the linear models, in the shape an MCS or
  Diebold-Mariano test wants. The deep models' predictions are in
  `results/<setting>/pred.npy` per seed.
* **Artifacts.** Logs `benchmark_h1_h5/logs/`, per-config json
  `benchmark_h1_h5/runs/`, HAR-RV diagnostics and figures
  `benchmark_h1_h5/har_rv/<PAIR>/HAR-RV results/`.